## Slurm Files, and Using the HPC Cluster

All of the steps I've taken so far, from downloading raw data to the PCA, have been performed on one sample from the knockout group (SRR36691286). But to recreate the actual analysis I need to use data from all four samples, and compare gene expression in different treatment groups. 

To download and analyze this massive amount of data, I requested an account on my lab's HPC cluster. Now that I got it set up I can rerun the downloading, quality control, and alignment for all four samples at once! 

Jobs for the HPC are submitted as Slurm files, which I'll save in a new folder alongside my original bash scripts.

### SBATCH Directives

At the top of each of these Slurm files is instructions called SBATCH directives. A set of instructions like this will be at the top of each file:

In [ ]:
#!/bin/tcsh
#SBATCH --job-name=ptp1b_job_    # Name for the job
#SBATCH --nodes=1                # Number of HPC nodes to use
#SBATCH --ntasks=1               # Number of tasks
#SBATCH --cpus-per-task=1        # Number of CPUs to use per task
#SBATCH --mem=1gb                # Amount of memory to use
#SBATCH --time=10:00:00          # Amount of time to give for the job
#SBATCH -e ptp1b_job.err         # Name for error file
#SBATCH -o ptp1b_job.out         # Name for output file

For each of my scripts the directives will look something like this, using my SRAToolkit script as an example:

In [ ]:
#!/bin/bash
#SBATCH --job-name=ptp1b_download
#SBATCH --nodes=1                                 # Nothing I'm doing requires more than 1 node
#SBATCH --ntasks=1                                # One task/process
#SBATCH --cpus-per-task=4                         # 4 CPUs is a reasonably small amount to request
#SBATCH --mem=16gb                                # 16gb memory is also reasonable
#SBATCH --time=10:00:00                           # This definitely won't take 10 hours, but just in case
#SBATCH -e ~/ptp1b_ad/logs/ptp1b_download.err     # Custom name and filepath for any error or output files
#SBATCH -o ~/ptp1b_ad/logs/ptp1b_download.out

After this I will remake each of my scripts, but expanding them to incorporate all four samples at once! This starts with downloading all four SRA accessions.

### Downloading

There are a few changes to using SRA Toolkit to download fastq files, relative to my first script.

To start, I have to load modules on the HPC. EB5 is a package with a lot of common scientific tools on it, and I'll also load SRA Toolkit by name to be safe:

In [ ]:
module load EB5
module load SRA-Toolkit

The instructions on my lab's HPC website recommend prefetching the samples, downloading them in the compressed .sra format before using fasterq-dump:

In [ ]:
prefetch -O ~/ptp1b_ad/data/sra SRR36691286 SRR36691287 SRR36691288 SRR36691289
# -O: Output location (new folder for sra files)

I also need to make sure this runs in the directory for this project, and that the directories for data and fastq files exist.

In [ ]:
# Start in the main project directory
cd ~/ptp1b_ad
# "~/": Absolute path starting from home directory

# Make data and fastq file directories
mkdir -p data/fastq
# -p: Create parent directory ('data') alongside new directory ('fastq') if necessary

Last is a set of instructions running the SRA Toolkit on each of the four files. Example for one file:

In [ ]:
# Run fasterq
fasterq-dump -e 4 -p -O data/fastq data/sra/SRR36691286/SRR36691286.sra

# Rename the output files based on treatment group
mv data/fastq/SRR36691286_1.fastq data/fastq/KO_SRR36691286_1.fastq
mv data/fastq/SRR36691286_2.fastq data/fastq/KO_SRR36691286_2.fastq

# Zip the files
gzip data/fastq/KO_SRR36691286_1.fastq
gzip data/fastq/KO_SRR36691286_2.fastq

### FastQC

Preparing a Slurm script for FastQC is pretty similar. It involves the module for FastQC, the same starting directory, and a new directory for output html files:

In [ ]:
module load EB5
module load FastQC

cd ~/ptp1b_ad
mkdir -p data/fastqc

After that it is easy to run on each sample:

In [ ]:
fastqc data/fastq/KO_SRR36691286_1.fastq.gz data/fastq/KO_SRR36691286_2.fastq.gz -o data/fastqc
fastqc data/fastq/KO_SRR36691287_1.fastq.gz data/fastq/KO_SRR36691287_2.fastq.gz -o data/fastqc
fastqc data/fastq/AD_SRR36691288_1.fastq.gz data/fastq/AD_SRR36691288_2.fastq.gz -o data/fastqc
fastqc data/fastq/AD_SRR36691289_1.fastq.gz data/fastq/AD_SRR36691289_2.fastq.gz -o data/fastqc

### Quantification

Now that I'm on the HPC, I can switch to the exact alignment that was used in the paper - the CellRanger tool. This was too intense to run on my own laptop. CellRanger takes a lot of the steps I had to do and turns them into one easy command.

I need a different reference genome than the one I used, a pre-built CellRanger reference package. Remember that in the supplementary info section the reference genome was given as:
- gex-mm10-2020-A, 10X Genomics

This can be downloaded straight off the 10X Genomics website onto the cluster and unpacked:

In [ ]:
wget https://cf.10xgenomics.com/supp/cell-exp/refdata-gex-mm10-2020-A.tar.gz

tar -xzf refdata-gex-mm10-2020-A.tar.gz
# tar: Command for manipulating archives
# -x: Extract the files
# -z: Uncompress the file with gzip
# -f refdata-gex-mm10-2020-A.tar.gz: Specified file name 

Now for the new slurm, to be extra safe I added extra memory/cores:

In [ ]:
#SBATCH --cpus-per-task=8
#SBATCH --mem=64gb
#SBATCH --time=24:00:00

Loading modules, and creating new output directories:

In [ ]:
module load EB5
module load CellRanger/9.0.1

cd ~/ptp1b_ad
mkdir -p results/cellranger_output/KO_SRR36691286
mkdir -p results/cellranger_output/KO_SRR36691287
mkdir -p results/cellranger_output/AD_SRR36691288
mkdir -p results/cellranger_output/AD_SRR36691289

Running CellRanger on one sample:

In [ ]:
cellranger count \
  --id=KO_SRR36691286 \
  --transcriptome=data/references/refdata-gex-mm10-2020-A \
  --fastqs=data/fastq \
  --sample=KO_SRR36691286 \
  --localcores=8 \
  --localmem=64 \
  --output-dir=results/cellranger_output/KO_SRR36691286

Ready for the next step!

### Quantification

This is the step with the most commands, but luckily I have already done a lot of the work. These reference files from earlier can easily be sent over to the HPC:
- The index (mm10.idx) generated from the reference transcriptome
- A whitelist of barcodes for 10xv3 sequencing (3M-february-2018.txt)
- A file with the names of all transcripts in the reference transcriptome (transcripts.txt)
- A file mapping all of the transcripts to genes (transcripts_to_genes.txt)

With these files already in hand, I only need to run the main steps on all four samples - pseudoalignment, sorting/ordering the barcodes, and making an expression matrix.

The fastq files need to be renamed to work with CellRanger, following a convention that specifies which lanes/reads they come from:

In [ ]:
mv KO_SRR36691286_1.fastq.gz KO_SRR36691286_S1_L001_R1_001.fastq.gz
mv KO_SRR36691286_2.fastq.gz KO_SRR36691286_S1_L001_R2_001.fastq.gz

Since CellRanger is very intensive, I requested some more memory and gave the job a larger time limit. A couple of the sbatch directives are different here:

In [ ]:
#SBATCH --mem=32gb
#SBATCH --time=24:00:00

To start I'll load the modules and make sure a directory is created to store results for each sample:

In [ ]:
module load EB5
module load kallisto
module load bustools

cd ~/ptp1b_ad
mkdir -p results/cellranger_output/KO_SRR36691286
mkdir -p results/cellranger_output/KO_SRR36691287
mkdir -p results/cellranger_output/AD_SRR36691288
mkdir -p results/cellranger_output/AD_SRR36691289

CellRanger is run in just one command that goes through the alignment, sorting/correcting, and creation of the matrix that I needed kallisto and bustools do perform earlier. There are a bunch of flags that specify how I want things done:

In [ ]:
cellranger count \
  --id=KO_SRR36691286 \
  --transcriptome=data/references/refdata-gex-mm10-2020-A \
  --fastqs=data/fastq \
  --sample=KO_SRR36691286 \
  --localcores=8 \
  --localmem=64 \
  --output-dir=results/cellranger_output/KO_SRR36691286 \
  --create-bam false
# cellranger count: Gene expression counting pipeline
# --id: Name for this run, used for its output folder
# --transcriptome: Path to reference genome with annotations
# --fastqs: Path to the input fastq files
# --sample: Name of the sample in the fastq files
# --localcores: Max number of CPUs to use
# --localmem: Max amount of memory to use (GB)
# --output-dir: Specify where to write output files
# --create-bam: Create BAM file (needed for some analysis like variant calling)

This CellRanger output is a matrix of gene/cell counts for each of the four samples.

### CellBender - cleaning the matrices

There is one last thing that was done in the paper before bringing the data into R for analysis. The matrices were cleaned up, removing some empty droplets and ambient (free-floating / not from a cell) RNA.

This was done with a program called CellBender. I'll create one last slurm file and run this on all four samples.

First on the HPC I need to create a conda environment that has CellBender available:

In [ ]:
# Loading conda module
module load EBModules
module load Anaconda3

# Creating new conda environment
conda create -n cellbender
# Activating envrionment and installing CellBender here
conda init
conda activate cellbender
pip install cellbender

Now I can run it on each sample's raw matrix from CellRanger, like this:

In [ ]:
cellbender remove-background \
    --input ~/ptp1b_ad/results/cellranger_output/KO_SRR36691286/outs/raw_feature_bc_matrix.h5/ \
    --output ~/ptp1b_ad/results/cellbender_output/KO_SRR36691286_cellbender.h5 \
    --epochs 150 \
    --cpu-threads 8

The next step is bringing this cleaned up data into R and repeating the analysis with Seurat to identify genes and cell types. The final goal is to compare expression between the two treatment groups.